# 04: プロンプトテンプレートの作成と MCP 接続確認

このノートブックでは、エージェントが使用するプロンプトテンプレートを DataRobot に作成し、MCP サーバーとの接続を確認します。

## 処理の流れ
1. 環境変数の読み込みと DataRobot クライアントの初期化
2. プロンプトテンプレートの作成
3. プロンプトテンプレートバージョンの作成
4. MCP デプロイメントの接続確認
5. 全環境変数のまとめと `.env` サンプル出力

## 1. 環境変数の読み込み

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv("../.env", override=True)

DATAROBOT_API_TOKEN = os.environ["DATAROBOT_API_TOKEN"]
DATAROBOT_ENDPOINT = os.environ.get("DATAROBOT_ENDPOINT", "https://app.datarobot.com/api/v2")

# 前のノートブックで設定した値
FORECAST_DEPLOYMENT_ID = os.environ.get("FORECAST_DEPLOYMENT_ID", "")
SCORING_DATASET_ID = os.environ.get("SCORING_DATASET_ID", "")
VDB_DEPLOYMENT_ID = os.environ.get("VDB_DEPLOYMENT_ID", "")
COMPANY_NAME = os.environ.get("COMPANY_NAME", "小売EC需要予測デモ")
MCP_DEPLOYMENT_ID = os.environ.get("MCP_DEPLOYMENT_ID", "")

print(f"Endpoint:              {DATAROBOT_ENDPOINT}")
print(f"Token:                 {DATAROBOT_API_TOKEN[:8]}...")
print(f"FORECAST_DEPLOYMENT_ID: {FORECAST_DEPLOYMENT_ID or '(未設定)'}")
print(f"SCORING_DATASET_ID:     {SCORING_DATASET_ID or '(未設定)'}")
print(f"VDB_DEPLOYMENT_ID:      {VDB_DEPLOYMENT_ID or '(未設定)'}")
print(f"MCP_DEPLOYMENT_ID:      {MCP_DEPLOYMENT_ID or '(未設定)'}")
print(f"COMPANY_NAME:           {COMPANY_NAME}")

## 2. DataRobot クライアントの初期化

In [ ]:
import datarobot as dr
import requests

client = dr.Client(
    token=DATAROBOT_API_TOKEN,
    endpoint=DATAROBOT_ENDPOINT,
)
print(f"DataRobot client initialized (v{dr.__version__})")

API_BASE = DATAROBOT_ENDPOINT
HEADERS = {
    "Authorization": f"Bearer {DATAROBOT_API_TOKEN}",
    "Content-Type": "application/json",
}

## 3. プロンプトテンプレートの作成

エージェントが使用するシステムプロンプトを DataRobot のプロンプトテンプレートとして作成します。

プロンプトには以下の変数を含めます:
- `{company_name}`: エージェントの表示名
- `{forecast_deployment_id}`: 時系列予測モデルのデプロイメントID
- `{scoring_dataset_id}`: スコアリング用データセットID

In [ ]:
SYSTEM_PROMPT_TEXT = """あなたは「{company_name}」の AI アシスタントです。
小売・EC業界の需要予測と市場分析を専門としています。

## あなたが使用できるツールカテゴリ

### 1. 時系列需要予測ツール
DataRobot の時系列予測モデル (Deployment ID: {forecast_deployment_id}) を使って、
店舗種別ごとの売上予測を行います。
- スコアリングデータセット ID: {scoring_dataset_id}
- 予測対象: sales_billion_yen (売上高・億円)
- 予測期間: 3ヶ月先まで
- シリーズ: store_type (店舗種別)

予測を実行する際は、デプロイメントIDとスコアリングデータセットIDを使用して
DataRobot Predictions API を呼び出してください。

### 2. EC市場調査レポート検索ツール (RAG)
経済産業省の EC 市場調査レポートをベースにしたベクターデータベースから、
関連情報を検索・取得できます。
- EC市場の最新動向
- BtoC、BtoB、CtoC の市場規模
- 業種別の EC 化率
- 越境ECの状況

### 3. MCP ツール
MCP (Model Context Protocol) サーバー経由で提供される外部ツールを使用できます。
利用可能なツールは動的に読み込まれます。

## 回答のガイドライン
- 数値データを含む回答では、出典を明示してください
- 予測結果を提示する際は、モデルの精度情報も合わせて伝えてください
- 日本語で丁寧に回答してください
- グラフや表が有効な場合は、Markdown 形式で表を作成してください
- 不確実な情報には適切な注釈を付けてください
"""

print("システムプロンプト (先頭500文字):")
print(SYSTEM_PROMPT_TEXT[:500])
print("...")

In [ ]:
# プロンプトテンプレートを作成
PROMPT_TEMPLATE_NAME = "小売EC需要予測エージェント プロンプト"

# 既存のプロンプトテンプレートを検索
resp = requests.get(
    f"{API_BASE}/genai/promptTemplates/",
    headers=HEADERS,
    params={"limit": 100},
)
resp.raise_for_status()
templates_data = resp.json()
existing_templates = [
    t for t in templates_data.get("data", [])
    if t.get("name") == PROMPT_TEMPLATE_NAME
]

if existing_templates:
    template = existing_templates[0]
    PROMPT_TEMPLATE_ID = template["id"]
    print(f"既存のプロンプトテンプレートを使用します: {template['name']}")
    print(f"  ID: {PROMPT_TEMPLATE_ID}")
else:
    create_resp = requests.post(
        f"{API_BASE}/genai/promptTemplates/",
        headers=HEADERS,
        json={
            "name": PROMPT_TEMPLATE_NAME,
            "description": "小売EC需要予測エージェント用のシステムプロンプト。時系列予測、RAG、MCPツールの3カテゴリをサポート。",
        },
    )
    create_resp.raise_for_status()
    template = create_resp.json()
    PROMPT_TEMPLATE_ID = template["id"]
    print(f"プロンプトテンプレートを作成しました: {template['name']}")
    print(f"  ID: {PROMPT_TEMPLATE_ID}")

## 4. プロンプトテンプレートバージョンの作成

テンプレートに対してバージョンを追加します。バージョン管理により、プロンプトの変更履歴を追跡できます。

In [ ]:
# バージョンを作成
version_resp = requests.post(
    f"{API_BASE}/genai/promptTemplates/{PROMPT_TEMPLATE_ID}/versions/",
    headers=HEADERS,
    json={
        "systemPrompt": SYSTEM_PROMPT_TEXT,
        "description": "初期バージョン: 3ツールカテゴリ (時系列予測、RAG、MCP) 対応",
    },
)
version_resp.raise_for_status()
version_data = version_resp.json()

print(f"プロンプトテンプレートバージョンを作成しました。")
print(f"  Version ID: {version_data.get('id', 'N/A')}")
print(f"  Template ID: {PROMPT_TEMPLATE_ID}")

## 5. MCP デプロイメントの接続確認

`.env` に `MCP_DEPLOYMENT_ID` が設定されている場合、デプロイメントの状態を確認します。

In [ ]:
if MCP_DEPLOYMENT_ID:
    print(f"MCP デプロイメント ID: {MCP_DEPLOYMENT_ID}")
    try:
        mcp_deployment = dr.Deployment.get(MCP_DEPLOYMENT_ID)
        print(f"  ラベル: {mcp_deployment.label}")
        print(f"  ステータス: {mcp_deployment.status}")
        print(f"  モデル ID: {mcp_deployment.model.get('id', 'N/A')}")

        # サービスヘルスチェック
        service_stats = mcp_deployment.get_service_stats()
        print(f"  サービス統計: {service_stats}")
        print("\nMCP デプロイメントへの接続に成功しました。")
    except Exception as e:
        print(f"  警告: MCP デプロイメントの確認に失敗しました: {e}")
        print("  デプロイメントが存在し、アクティブであることを確認してください。")
else:
    print("MCP_DEPLOYMENT_ID が設定されていません。")
    print("MCP ツールを使用しない場合は、この手順をスキップできます。")
    print("MCP ツールが必要な場合は、デプロイ後に .env に MCP_DEPLOYMENT_ID を設定してください。")

## 6. 全環境変数のまとめ

セットアップで取得した全ての ID をまとめて表示します。

In [ ]:
print("=" * 70)
print("セットアップ完了: 全環境変数のまとめ")
print("=" * 70)

env_vars = {
    "DATAROBOT_API_TOKEN": DATAROBOT_API_TOKEN[:8] + "...(セキュリティのため省略)",
    "DATAROBOT_ENDPOINT": DATAROBOT_ENDPOINT,
    "PROMPT_TEMPLATE_ID": PROMPT_TEMPLATE_ID,
    "FORECAST_DEPLOYMENT_ID": FORECAST_DEPLOYMENT_ID or "(未設定 - 02_build_timeseries_model.ipynb を実行してください)",
    "SCORING_DATASET_ID": SCORING_DATASET_ID or "(未設定 - 02_build_timeseries_model.ipynb を実行してください)",
    "VDB_DEPLOYMENT_ID": VDB_DEPLOYMENT_ID or "(未設定 - 03_create_vdb.ipynb を実行してください)",
    "COMPANY_NAME": COMPANY_NAME,
    "MCP_DEPLOYMENT_ID": MCP_DEPLOYMENT_ID or "(オプション - 必要に応じて設定)",
}

for key, value in env_vars.items():
    status = "OK" if value and not value.startswith("(") else "--"
    print(f"  [{status}] {key}={value}")

## 7. `.env` サンプルファイルの出力

以下の内容をコピーして `.env` ファイルに追記してください。

In [ ]:
env_content = f"""# === 小売EC需要予測エージェント設定 ===
# DataRobot Prompt Template ID (バージョン管理されたシステムプロンプト)
PROMPT_TEMPLATE_ID={PROMPT_TEMPLATE_ID}

# 時系列予測モデルのデプロイメントID
FORECAST_DEPLOYMENT_ID={FORECAST_DEPLOYMENT_ID}

# スコアリング用データセットID
SCORING_DATASET_ID={SCORING_DATASET_ID}

# ベクターDB (PDF RAG) のデプロイメントID
VDB_DEPLOYMENT_ID={VDB_DEPLOYMENT_ID}

# エージェント表示名
COMPANY_NAME={COMPANY_NAME}

# MCP server configurations
MCP_SERVER_PORT=9000
MCP_DEPLOYMENT_ID={MCP_DEPLOYMENT_ID}
"""

print("=" * 70)
print(".env ファイルに追記する内容:")
print("=" * 70)
print(env_content)
print("=" * 70)

In [ ]:
# 必要に応じて .env ファイルに自動追記 (確認後にコメント解除して実行)
# import pathlib
# env_path = pathlib.Path("../.env").resolve()
# with open(env_path, "a") as f:
#     f.write("\n" + env_content)
# print(f".env ファイルに追記しました: {env_path}")

print("セットアップが完了しました。")
print("上記の環境変数を .env に設定した後、エージェントを起動できます。")